# Bhagavad Gita Knowledge Graph — Loader

Loads chapters, verses, speakers, addressees, epithets, setting, and lemmatized terms
from the English translations into the local Neo4j `TheGitaProject` database.

- **Spec:** `docs/superpowers/specs/2026-08-20-gita-knowledge-graph-design.md`
- Deterministic + idempotent: re-running rebuilds the graph with no duplicates.
- Requires a local Neo4j with a `TheGitaProject` database and a `gita-knowledge-graph/.env`
  (copy from `.env.example`).

## 1. Config & connect

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
PKG = ROOT / "gita-knowledge-graph"
sys.path.insert(0, str(PKG))  # make gita_kg importable regardless of cwd

from dotenv import load_dotenv
from neo4j import GraphDatabase

import gita_kg as gk

load_dotenv(PKG / ".env")
cfg = gk.load_config()
driver = GraphDatabase.driver(cfg.uri, auth=(cfg.user, cfg.password))
driver.verify_connectivity()
print("connected:", cfg.uri, "->", cfg.database)

## 2. spaCy pipeline (EntityRuler for epithets)

In [ ]:
import spacy

nlp = gk.build_epithet_ruler(spacy.load("en_core_web_sm"))

## 3. Parse the verses

In [ ]:
import collections

VERSES_DIR = ROOT / "data/TheGitaProject/Verses"
records = gk.build_records(VERSES_DIR, nlp)
print(f"parsed {len(records)} verses")
print("speakers:", collections.Counter(r.speaker for r in records))

## 4. Load into Neo4j (constraints → seeds → verses)

In [ ]:
def run_ops(ops):
    with driver.session(database=cfg.database) as session:
        for cypher, params in ops:
            session.run(cypher, **params)


run_ops(gk.constraint_ops())
run_ops(gk.seed_ops())
run_ops(gk.verse_ops(records))
print("load complete")

## 5. Verification

In [ ]:
def one(cypher):
    with driver.session(database=cfg.database) as s:
        return s.run(cypher).single()[0]


print("verses:", one("MATCH (v:Verse) RETURN count(v)"))
print(
    "no SPOKEN_BY:",
    one("MATCH (v:Verse) WHERE NOT (v)-[:SPOKEN_BY]->() RETURN count(v)"),
)
print(
    "no ADDRESSED_TO:",
    one("MATCH (v:Verse) WHERE NOT (v)-[:ADDRESSED_TO]->() RETURN count(v)"),
)
print("terms:", one("MATCH (t:Term) RETURN count(t)"))
print("epithet edges:", one("MATCH ()-[r:USES_EPITHET]->() RETURN count(r)"))

In [ ]:
with driver.session(database=cfg.database) as s:
    rows = s.run(
        "MATCH (v:Verse)-[:SPOKEN_BY]->(:Person {name:'Arjuna'}) "
        "RETURN v.id AS id ORDER BY v.chapter, v.verse LIMIT 10"
    ).values()
print("Arjuna's first verses:", rows)

In [ ]:
with driver.session(database=cfg.database) as s:
    rows = s.run(
        "MATCH (:Chapter {number:2})-[:HAS_VERSE]->(v)-[m:MENTIONS_TERM]->(t) "
        "RETURN t.lemma AS term, sum(m.count) AS n ORDER BY n DESC LIMIT 10"
    ).values()
print("Chapter 2 top terms:", rows)

## 6. Close

In [ ]:
driver.close()